# Data preparation — justification for grouping leading vehicles into 5 classes

Same battery as the target (PR/BTW) grouping, applied to the **leading vehicle** classes,
on **leading-vehicle speed** (the grouping basis) and **time headway**:

Kruskal–Wallis omnibus + **Dunn–Holm** post-hoc; **Cliff's δ** (effect size); two-sample
**KS**; **TOST** equivalence; and a JZS **Bayes factor BF01** (evidence for equivalence).
Graphics: KDE + overlaid histograms, and empirical CDFs with pairwise KS panels.

Logic: sub-classes merged into the **same** leading class should be equivalent
(Dunn n.s., δ negligible, TOST/BF01 favour equivalence); **different** classes should
differ (significant, large δ, non-equivalent).

**Configure Cell 1.** `data3.xlsx` holds the merged 5-class `V_Leading_Class`; point
`RAW_LEADING_COL` at your granular leading-type column and fill `GROUP_MAP` to justify the
merges. Left at the identity default, the notebook instead demonstrates that the 5 classes
are pairwise distinct.

In [1]:
# --- Cell 1: Imports, paths, CONFIGURATION ---
import os
import numpy as np
import pandas as pd
from scipy import stats
from scipy.integrate import quad

BASE     = r"D:\Headway"
DATA     = os.path.join(BASE, "data3.xlsx")
TABLES   = os.path.join(BASE, "Tables")
GRAPHICS = os.path.join(BASE, "Graphics")
os.makedirs(TABLES, exist_ok=True); os.makedirs(GRAPHICS, exist_ok=True)

# ---- CONFIGURE ----
# Granular leading-type column to justify merging. Defaults to the merged 5-class column.
RAW_LEADING_COL = "V_Leading_Class"
# Map each granular leading sub-type -> its intended class. EDIT for your sub-types, e.g.
# {"Car":"4W","Jeep":"4W","CNG":"MT_3W","Rickshaw":"NMT_3W","Bike":"MT_2W","Bicycle":"NMT_2W"}
# Identity default (each class maps to itself) so it runs out-of-the-box.
GROUP_MAP = {"4W":"4W","MT_3W":"MT_3W","NMT_3W":"NMT_3W","MT_2W":"MT_2W","NMT_2W":"NMT_2W"}

# Metrics: leading speed is the grouping basis; headway included as the outcome.
LEAD_SPEED = "Leading_Speed_km/hr"
METRICS = {"Leading speed (km/h)": LEAD_SPEED, "Time headway (s)": "Time_Headway"}

# Equivalence margins (metric units). None -> 0.2 x pooled SD ("small"). Set a domain value
# and keep it consistent with the target-grouping analysis.
EQUIV_MARGIN = {"Leading speed (km/h)": None, "Time headway (s)": None}
BF_R = 0.707     # JZS prior scale (Rouder et al. 2009 default)
ALPHA = 0.05

In [2]:
# --- Cell 2: Load, resolve columns, assign groups ---
df = pd.read_excel(DATA)
if RAW_LEADING_COL not in df.columns:
    raise ValueError(f"'{RAW_LEADING_COL}' not in columns: {list(df.columns)}")

present = [c for c in df[RAW_LEADING_COL].dropna().unique()]
classes = [c for c in present if c in GROUP_MAP]
unmapped = [c for c in present if c not in GROUP_MAP]
if unmapped: print("WARNING: not in GROUP_MAP (ignored):", unmapped)
GROUP_OF = {c: GROUP_MAP[c] for c in classes}
# order sub-classes by their group then name for tidy tables
classes = sorted(classes, key=lambda c: (GROUP_OF[c], str(c)))
print(f"Leading column: {RAW_LEADING_COL} | sub-classes: {classes}")
for g in sorted(set(GROUP_OF.values())):
    members = [c for c in classes if GROUP_OF[c] == g]
    print(f"   {g}: {members}  n={[int((df[RAW_LEADING_COL]==m).sum()) for m in members]}")

Leading column: V_Leading_Class | sub-classes: ['4W', 'MT_2W', 'MT_3W', 'NMT_2W', 'NMT_3W']
   4W: ['4W']  n=[293]
   MT_2W: ['MT_2W']  n=[96]
   MT_3W: ['MT_3W']  n=[290]
   NMT_2W: ['NMT_2W']  n=[51]
   NMT_3W: ['NMT_3W']  n=[168]


In [3]:
# --- Cell 3: Statistical helpers (incl. JZS Bayes factor) ---
def ecdf(x): xs = np.sort(x); return xs, np.arange(1, len(xs)+1)/len(xs)
def kde_curve(x, grid):
    return stats.gaussian_kde(x)(grid) if (len(np.unique(x)) >= 3 and len(x) >= 5) else None

def kruskal_eps(groups):
    H, p = stats.kruskal(*groups); N = sum(len(g) for g in groups); eps2 = H/(N-1)
    mag = ("negligible" if eps2<0.01 else "small" if eps2<0.06 else "medium" if eps2<0.14 else "large")
    return H, p, eps2, mag, N

def dunn_holm(data_by_class, group_of):
    labels = list(data_by_class.keys())
    data = np.concatenate([np.asarray(data_by_class[l]) for l in labels])
    grp  = np.concatenate([[l]*len(data_by_class[l]) for l in labels])
    N = len(data); ranks = stats.rankdata(data)
    _, cnt = np.unique(data, return_counts=True); ties = np.sum(cnt**3 - cnt)
    sigma2 = (N*(N+1)/12.0) - ties/(12.0*(N-1))
    mrank = {l: ranks[grp==l].mean() for l in labels}; n = {l:int((grp==l).sum()) for l in labels}
    rows = []
    for i in range(len(labels)):
        for j in range(i+1, len(labels)):
            a, b = labels[i], labels[j]
            z = (mrank[a]-mrank[b]) / np.sqrt(sigma2*(1.0/n[a]+1.0/n[b]))
            comp = "within" if group_of[a]==group_of[b] else "between"
            rows.append([a, b, comp, n[a], n[b], round(z,3), 2*stats.norm.sf(abs(z))])
    ps = np.array([r[6] for r in rows]); m=len(ps); order=np.argsort(ps); prev=0.0; adj=np.empty(m)
    for k, idx in enumerate(order):
        prev = max(prev, (m-k)*ps[idx]); adj[idx] = min(prev, 1.0)
    for r, a in zip(rows, adj): r[6] = round(r[6],4); r.append(round(a,4))
    return rows  # [a,b,comp,n_a,n_b,z,p_raw,p_holm]

def cliffs_delta(x, y):
    x=np.asarray(x); y=np.asarray(y); d=float(np.sign(x[:,None]-y[None,:]).mean()); a=abs(d)
    mag=("negligible" if a<0.147 else "small" if a<0.33 else "medium" if a<0.474 else "large")
    return round(d,4), mag

def tost(x, y, margin):
    x=np.asarray(x); y=np.asarray(y); d=x.mean()-y.mean()
    vx, vy = x.var(ddof=1)/len(x), y.var(ddof=1)/len(y); se=np.sqrt(vx+vy)
    dfw=(vx+vy)**2/(vx**2/(len(x)-1)+vy**2/(len(y)-1))
    p_low=stats.t.sf((d+margin)/se, dfw); p_up=stats.t.cdf((d-margin)/se, dfw)
    return round(max(p_low, p_up), 4)

def bf01_jzs(x, y, r=BF_R):
    """JZS Bayes factor for equivalence (BF01 = 1/BF10), Rouder et al. (2009)."""
    nx, ny = len(x), len(y)
    t = stats.ttest_ind(x, y, equal_var=True).statistic
    n = nx*ny/(nx+ny); dfree = nx+ny-2
    integrand = lambda g: ((1+n*g*r**2)**(-0.5) *
                           (1+t**2/((1+n*g*r**2)*dfree))**(-(dfree+1)/2) *
                           (2*np.pi)**(-0.5) * g**(-1.5) * np.exp(-1/(2*g)))
    bf10 = quad(integrand, 0, np.inf)[0] / ((1+t**2/dfree)**(-(dfree+1)/2))
    return 1.0/bf10 if bf10 > 0 else np.inf

In [4]:
# --- Cell 4: Run all tests per metric; assemble tables ---
def pfmt(p): return "<0.001" if p < 1e-3 else round(p, 4)
def bfmt(bf): return "<0.01" if bf < 0.01 else (">100" if bf > 100 else round(bf, 2))

long_rows, summary_rows, omnibus_rows = [], [], []
for mname, mcol in METRICS.items():
    sub = df[[RAW_LEADING_COL, mcol]].dropna(); sub = sub[sub[RAW_LEADING_COL].isin(classes)]
    dbc = {c: sub.loc[sub[RAW_LEADING_COL]==c, mcol].values for c in classes}
    margin = EQUIV_MARGIN[mname] if EQUIV_MARGIN.get(mname) else 0.2*sub[mcol].std()

    H, p, eps2, mag, N = kruskal_eps(list(dbc.values()))
    omnibus_rows.append(dict(Metric=mname, k=len(classes), N=N, H=round(H,2),
                             p=pfmt(p), epsilon2=round(eps2,3), magnitude=mag))
    summary_rows.append(dict(Comparison=f"{mname}: Kruskal-Wallis H = {H:.2f}, "
                                        f"p {('<0.001' if p<1e-3 else '= '+str(round(p,4)))}, "
                                        f"eps2 = {eps2:.3f}",
                             Dunn_z="", p_Holm="", Cliffs_delta="", KS_D="", p_KS="",
                             TOST_p="", BF01=""))
    dunn = {(r[0], r[1]): r for r in dunn_holm(dbc, GROUP_OF)}
    for (a, b), r in dunn.items():
        _, _, comp, na, nb, z, p_raw, p_holm = r
        cd, cmag = cliffs_delta(dbc[a], dbc[b])
        D, pk = stats.ks_2samp(dbc[a], dbc[b])
        tp = tost(dbc[a], dbc[b], margin); bf = bf01_jzs(dbc[a], dbc[b])
        long_rows.append(dict(Metric=mname, class_A=a, class_B=b, comparison=comp,
                              n_A=na, n_B=nb, Dunn_z=z, p_Holm=p_holm,
                              Cliffs_delta=cd, delta_mag=cmag, KS_D=round(D,3), p_KS=pfmt(pk),
                              TOST_p=tp, BF01=round(bf,3)))
        summary_rows.append(dict(Comparison=f"{a} vs {b}", Dunn_z=z, p_Holm=p_holm,
                                 Cliffs_delta=cd, KS_D=round(D,3), p_KS=pfmt(pk),
                                 TOST_p=tp, BF01=bfmt(bf)))

omnibus_kw   = pd.DataFrame(omnibus_rows)
pairwise_long= pd.DataFrame(long_rows)
summary_table= pd.DataFrame(summary_rows)[["Comparison","Dunn_z","p_Holm","Cliffs_delta",
                                           "KS_D","p_KS","TOST_p","BF01"]]
summary_table

,Comparison,Dunn_z,p_Holm,Cliffs_delta,KS_D,p_KS,TOST_p,BF01
0,Leading speed (km/h): Kruskal-Wallis H = 212.6...,,,,,,,
1,4W vs MT_2W,-3.175,0.006,-0.2439,0.204,0.0041,0.9847,<0.01
2,4W vs MT_3W,1.533,0.1252,0.0815,0.101,0.0904,0.2924,1.49
3,4W vs NMT_2W,3.15,0.006,0.2981,0.3,<0.001,0.9992,0.01
4,4W vs NMT_3W,12.347,0.0,0.6857,0.582,<0.001,1.0,<0.01
5,MT_2W vs MT_3W,4.25,0.0001,0.3209,0.282,<0.001,0.9996,<0.01
6,MT_2W vs NMT_2W,4.913,0.0,0.5116,0.455,<0.001,1.0,<0.01
7,MT_2W vs NMT_3W,12.258,0.0,0.7959,0.647,<0.001,1.0,<0.01
8,MT_3W vs NMT_2W,2.311,0.0417,0.2323,0.272,0.0025,0.9609,0.12
9,MT_3W vs NMT_3W,11.014,0.0,0.6402,0.544,<0.001,1.0,<0.01


In [5]:
# --- Cell 5: Save tables -> Excel ---
out_path = os.path.join(TABLES, "prep_leading_grouping.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as xl:
    summary_table.to_excel(xl,  sheet_name="Summary_table", index=False)
    pairwise_long.to_excel(xl,  sheet_name="Pairwise_long", index=False)
    omnibus_kw.to_excel(xl,     sheet_name="Omnibus_KW", index=False)
print("Saved:", out_path)

Saved: D:\Headway\Tables\prep_leading_grouping.xlsx


In [6]:
# --- Cell 6: KDE / ECDF / histogram data + native Excel charts (matplotlib-free) ---
from openpyxl import load_workbook
from openpyxl.chart import ScatterChart, Reference, Series
wb = load_workbook(out_path)
for mname, mcol in METRICS.items():
    sub = df[[RAW_LEADING_COL, mcol]].dropna(); sub = sub[sub[RAW_LEADING_COL].isin(classes)]
    grid = np.linspace(sub[mcol].min(), sub[mcol].max(), 200)
    ws = wb.create_sheet(("curv_"+mname[:20]).replace(" ","_").replace("(","").replace(")","").replace("/","_")[:31])
    ws.cell(1,1,"grid"); [ws.cell(i+2,1,float(grid[i])) for i in range(len(grid))]
    kde_ch = ScatterChart(); kde_ch.title=f"KDE - {mname}"; kde_ch.x_axis.delete=False; kde_ch.y_axis.delete=False
    ecdf_ch= ScatterChart(); ecdf_ch.title=f"ECDF - {mname}"; ecdf_ch.x_axis.delete=False; ecdf_ch.y_axis.delete=False
    col = 2
    for c in classes:
        x = sub.loc[sub[RAW_LEADING_COL]==c, mcol].values
        k = kde_curve(x, grid)
        if k is not None:
            ws.cell(1,col,f"kde_{c}"); [ws.cell(i+2,col,float(k[i])) for i in range(len(grid))]
            s=Series(Reference(ws,min_col=col,min_row=1,max_row=len(grid)+1),
                     Reference(ws,min_col=1,min_row=2,max_row=len(grid)+1),title_from_data=True)
            s.smooth=True; kde_ch.series.append(s); col+=1
        xs, ys = ecdf(x)
        ws.cell(1,col,f"ex_{c}"); ws.cell(1,col+1,f"ey_{c}")
        for i in range(len(xs)): ws.cell(i+2,col,float(xs[i])); ws.cell(i+2,col+1,float(ys[i]))
        s2=Series(Reference(ws,min_col=col+1,min_row=1,max_row=len(xs)+1),
                  Reference(ws,min_col=col,min_row=2,max_row=len(xs)+1),title_from_data=True)
        ecdf_ch.series.append(s2); col+=2
    kde_ch.height,kde_ch.width=9,16; ecdf_ch.height,ecdf_ch.width=9,16
    ws.add_chart(kde_ch,"B25"); ws.add_chart(ecdf_ch,"B46")
wb.save(out_path)
print("Embedded KDE/ECDF charts into:", out_path)

Embedded KDE/ECDF charts into: D:\Headway\Tables\prep_leading_grouping.xlsx


In [7]:
# --- Cell 7: Publication figures via matplotlib (KDE+hist, ECDF+pairwise KS) ---
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from itertools import combinations
    plt.rcParams.update({"font.size":9,"axes.grid":True,"grid.alpha":0.3,
                         "figure.dpi":300,"savefig.bbox":"tight"})
    palette = plt.cm.tab10(np.linspace(0,1,10))
    cmap = {c: palette[i % 10] for i, c in enumerate(classes)}

    for mname, mcol in METRICS.items():
        sub = df[[RAW_LEADING_COL, mcol]].dropna(); sub = sub[sub[RAW_LEADING_COL].isin(classes)]
        grid = np.linspace(sub[mcol].min(), sub[mcol].max(), 200)
        tag = mcol.replace("/", "_")

        # (1) density + histogram
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
        for c in classes:
            x = sub.loc[sub[RAW_LEADING_COL]==c, mcol].values; k = kde_curve(x, grid)
            if k is not None: ax[0].plot(grid, k, color=cmap[c], lw=1.6, label=str(c))
            ax[1].hist(x, bins=25, density=True, alpha=0.4, color=cmap[c], label=str(c))
        ax[0].set_title("(a) Kernel density"); ax[1].set_title("(b) Histograms")
        for a_ in ax: a_.set_xlabel(mname)
        ax[0].set_ylabel("Density"); ax[0].legend(frameon=False, fontsize=7)
        fig.suptitle(f"{mname} — density & histogram", fontsize=11)
        fig.savefig(os.path.join(GRAPHICS, f"prep_lead_density_{tag}.png")); plt.close(fig)

        # (2) all-group ECDF + pairwise KS grid
        pairs = list(combinations(classes, 2)); npr = len(pairs)
        ncol = 4; nrow = int(np.ceil((npr+1)/ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(3.1*ncol, 2.7*nrow)); axf = np.ravel(axes)
        for c in classes:
            xs, ys = ecdf(sub.loc[sub[RAW_LEADING_COL]==c, mcol].values)
            axf[0].step(xs, ys, where="post", color=cmap[c], lw=1.3, label=str(c))
        axf[0].set_title("(a) Empirical CDFs"); axf[0].set_ylabel("Cumulative prob."); axf[0].legend(frameon=False, fontsize=6)
        for idx, (a, b) in enumerate(pairs, start=1):
            xa = sub.loc[sub[RAW_LEADING_COL]==a, mcol].values
            xb = sub.loc[sub[RAW_LEADING_COL]==b, mcol].values
            D, pk = stats.ks_2samp(xa, xb)
            xs, ys = ecdf(xa); axf[idx].step(xs, ys, where="post", color=cmap[a], lw=1.3, label=str(a))
            xs, ys = ecdf(xb); axf[idx].step(xs, ys, where="post", color=cmap[b], lw=1.3, label=str(b))
            comp = "within" if GROUP_OF[a]==GROUP_OF[b] else "between"
            axf[idx].set_title(f"{a} vs {b} [{comp}]", fontsize=7)
            axf[idx].annotate(f"D={D:.3f}\np={'<0.001' if pk<1e-3 else round(pk,3)}",
                              xy=(0.55, 0.12), xycoords="axes fraction", fontsize=6)
            axf[idx].legend(frameon=False, fontsize=6)
        for a_ in axf[npr+1:]: a_.axis("off")
        for a_ in axf: a_.set_xlabel(mname, fontsize=7)
        fig.suptitle(f"{mname} — ECDF & KS", fontsize=11); fig.tight_layout()
        fig.savefig(os.path.join(GRAPHICS, f"prep_lead_ecdf_{tag}.png")); plt.close(fig)
    print("Saved leading-grouping figures to:", GRAPHICS)
except Exception as e:
    print("matplotlib unavailable (", type(e).__name__, ") - skipped PNGs; use Excel charts/data.")

Saved leading-grouping figures to: D:\Headway\Graphics
